# Variables nuevas para el modelo de zona-turno (issue #21)

**Proyecto integrador de tecnologías de Big Data** · Predicción de delitos en Chicago · Valentino De Paola, Eduardo López y Carlos David González

En la Fase 3 ningún modelo le ganó al **promedio histórico** de cada zona-turno. Con 2022 a 2025 (issue #20), Random Forest ya le gana a su propio promedio en 8 de 10 semanas, pero todo queda peor que con un año: los delitos bajaron en 2025 y promediar toda la historia sobreestima el final del año.

Este notebook le agrega al modelo información que el promedio histórico no tiene, en cinco grupos, y mide cuánto aporta cada uno:

| Grupo | Variables | ¿Mira solo el pasado? |
|---|---|---|
| Retrasos | Incidentes de la zona-turno hace 2 y hace 52 semanas, y su promedio de las últimas 52 semanas | Sí |
| Estacionalidad | Semana del año como seno y coseno | Sí (es el calendario) |
| Vecinos | Incidentes de la semana anterior en los 6 beats más cercanos, mismo turno | Sí |
| Festivos | Días festivos federales de la semana | Sí (se conocen de antemano) |
| Clima | Temperatura media, lluvia y nieve **de la semana que se predice** | **No**: es como tener un pronóstico perfecto. Se prueba aparte y se marca como optimista |

Parte de lo que ya está en HDFS: la tabla zona-turno de 2022 a 2025 (`chicago.zona_turno_2022_2025`, sección A3 de `fase3_pipeline.ipynb` con `ANIOS = [2022, 2023, 2024, 2025]`) y los incidentes limpios (`chicago.crimes_2022_2025`), para ubicar los beats. El clima lo deja en raw `scripts/ingesta-clima.sh`.

**Cómo correrlo:** en la instancia, con `LOCAL = False`, *Kernel → Restart Kernel and Run All Cells*. Antes tiene que haber corrido `fase3_pipeline.ipynb` con los cuatro años.

## Configuración

In [ ]:
LOCAL = False                 # False: instancia (HDFS + YARN) · True: prueba en una compu con los datos locales
FECHA_CLIMA = "2026-11-18"    # carpeta de la ingesta del clima en HDFS (la imprime scripts/ingesta-clima.sh)

PRIMER_ANIO, ANIO_PRUEBA = 2022, 2025
SEMANA_CORTE = 42             # del último año: las semanas 42 a 51 de 2025 prueban, como en la Fase 3
ETIQUETA = f"{PRIMER_ANIO}_{ANIO_PRUEBA}"

RUTA_DATALAKE = "../data" if LOCAL else "hdfs:///datalake"
RUTA_ZONA_TURNO = f"{RUTA_DATALAKE}/curated/zona_turno_{ETIQUETA}"
TABLA_ZONA_TURNO = f"chicago.zona_turno_{ETIQUETA}"
RUTA_CRIMENES = f"{RUTA_DATALAKE}/processed/crimes_{ETIQUETA}"
TABLA_CRIMENES = f"chicago.crimes_{ETIQUETA}"
RUTA_CLIMA = "../data/clima_ohare.csv" if LOCAL else f"hdfs:///datalake/raw/clima_ohare/fecha={FECHA_CLIMA}/clima_ohare.csv"

RUTA_SALIDA = f"{RUTA_DATALAKE}/curated/zona_turno_variables_{ETIQUETA}"
TABLA_SALIDA = f"chicago.zona_turno_variables_{ETIQUETA}"   # tabla zona-turno con todas las variables nuevas
RUTA_METRICAS = f"{RUTA_DATALAKE}/curated/metricas_variables_{ETIQUETA}"

In [ ]:
import math
import time
from datetime import date, timedelta

import matplotlib.pyplot as plt
import pandas as pd
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.sql import SparkSession, Window, functions as F

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

builder = (SparkSession.builder
           .appName(f"variables-nuevas-{'local' if LOCAL else 'cluster'}")
           .config("spark.sql.session.timeZone", "UTC"))

if LOCAL:
    spark = (builder.master("local[*]")
             .config("spark.driver.memory", "3g")
             .config("spark.ui.showConsoleProgress", "false")
             .getOrCreate())
else:
    spark = builder.enableHiveSupport().getOrCreate()   # master = yarn (spark-defaults.conf)

spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| master:", spark.sparkContext.master)


def a_semana_global(anio, semana):
    """Número de semana contado desde la semana 0 de 2022 (52 semanas por año), igual que en A3 de la Fase 3."""
    return (anio - PRIMER_ANIO) * 52 + semana


CORTE_GLOBAL = a_semana_global(ANIO_PRUEBA, SEMANA_CORTE)

## 1. Carga

La tabla zona-turno tiene una fila por beat × turno de 6 horas × semana, con los incidentes y las cuatro variables de la Fase 3. Empieza en la semana 4 de 2022, porque A3 necesita cuatro semanas previas.

In [ ]:
if LOCAL:
    zona_turno = spark.read.parquet(RUTA_ZONA_TURNO)
    crimenes = spark.read.parquet(RUTA_CRIMENES)
else:
    zona_turno = spark.table(TABLA_ZONA_TURNO)
    crimenes = spark.table(TABLA_CRIMENES)

zona_turno = zona_turno.drop("features", "conjunto").cache()   # el vector y la división se rehacen aquí
VARIABLES_FASE3 = ["incidentes_semana_anterior", "promedio_4_semanas", "promedio_historico", "incidentes_beat_semana_anterior"]

n_zona_turno = zona_turno.count()
n_zonas = zona_turno.select("beat", "turno").distinct().count()
semanas = zona_turno.agg(F.min("semana_global"), F.max("semana_global")).first()
n_semanas = semanas[1] - semanas[0] + 1
assert n_zona_turno == n_zonas * n_semanas, "Faltan semanas en alguna zona-turno: los retrasos por filas no servirían"
print(f"{n_zona_turno:,} filas · {n_zonas:,} zonas-turno · semanas {semanas[0]} a {semanas[1]} ({n_semanas} por zona-turno)")

## 2. Variables nuevas

### 2.1 Retrasos y promedio del último año

Todas se calculan con ventanas por zona-turno ordenadas por `semana_global`. Como cada zona-turno tiene todas sus semanas (lo comprueba la celda anterior), retrasar *k* filas es retrasar *k* semanas.

- `incidentes_hace_2`: la semana antes de la anterior (la anterior ya la tiene el modelo de la Fase 3).
- `incidentes_hace_52`: la misma semana del año pasado.
- `promedio_52_semanas`: el promedio de las últimas 52 semanas. A diferencia del promedio histórico, olvida los años viejos, así que sigue la baja de 2025 que encontró el issue #20.

In [ ]:
por_zona = Window.partitionBy("beat", "turno").orderBy("semana_global")
ultimo_anio = por_zona.rowsBetween(-52, -1)

variables = (zona_turno
             .withColumn("incidentes_hace_2", F.lag("incidentes", 2).over(por_zona))
             .withColumn("incidentes_hace_52", F.lag("incidentes", 52).over(por_zona))
             .withColumn("promedio_52_semanas", F.avg("incidentes").over(ultimo_anio))
             .withColumn("semanas_en_promedio_52", F.count("incidentes").over(ultimo_anio)))

### 2.2 Estacionalidad

La semana del año (0 a 51) entra como un punto en un círculo: seno y coseno de `2π · semana / 52`. Así la semana 51 queda junto a la 0, y no en el otro extremo como pasaría con el número solo.

En la Fase 3 la semana del año quedó fuera a propósito: con un solo año, las semanas de prueba caían en meses que el modelo nunca había visto. Con cuatro años el modelo ya vio tres otoños.

In [ ]:
angulo = 2 * math.pi * F.col("semana") / 52
variables = (variables
             .withColumn("semana_seno", F.sin(angulo))
             .withColumn("semana_coseno", F.cos(angulo)))

### 2.3 Beats vecinos

El centro de cada beat es la mediana de las coordenadas originales de sus incidentes (las imputadas se excluyen). Sus vecinos son los 6 beats con el centro más cercano. La distancia se calcula en kilómetros con una proyección plana, que a la escala de Chicago es suficiente: un grado de latitud mide 110.57 km y uno de longitud, 111.32 × cos(41.85°) ≈ 82.95 km.

`incidentes_vecinos_semana_anterior` es el promedio de incidentes de esos 6 beats, en el mismo turno, la semana anterior.

In [ ]:
VECINOS = 6
KM_LATITUD, KM_LONGITUD = 110.57, 111.32 * math.cos(math.radians(41.85))

beats_modelo = zona_turno.select("beat").distinct()
centros = (crimenes.filter(~F.col("coord_imputada") & F.col("latitude").isNotNull())
           .join(beats_modelo, "beat", "left_semi")
           .groupBy("beat")
           .agg(F.median("latitude").alias("lat"), F.median("longitude").alias("lon")))
assert centros.count() == beats_modelo.count(), "Algún beat del modelo no tiene centro"

a, b = centros.alias("a"), centros.alias("b")
cercania = Window.partitionBy("beat").orderBy("distancia_km", "vecino")
vecinos = (a.crossJoin(b).filter(F.col("a.beat") != F.col("b.beat"))
           .select(F.col("a.beat").alias("beat"), F.col("b.beat").alias("vecino"),
                   F.sqrt(((F.col("a.lat") - F.col("b.lat")) * KM_LATITUD) ** 2
                          + ((F.col("a.lon") - F.col("b.lon")) * KM_LONGITUD) ** 2).alias("distancia_km"))
           .withColumn("puesto", F.row_number().over(cercania))
           .filter(F.col("puesto") <= VECINOS)
           .cache())

distancias = vecinos.groupBy("puesto").agg(F.round(F.expr("percentile(distancia_km, 0.5)"), 2).alias("mediana (km)"),
                                           F.round(F.max("distancia_km"), 2).alias("máximo (km)")).orderBy("puesto").toPandas()
print(f"{vecinos.count():,} pares beat-vecino ({VECINOS} por beat)")
distancias.set_index("puesto")

In [ ]:
# Incidentes de cada vecino, corridos una semana: lo que pasó en el vecino en la semana anterior a la que se predice
vecino_anterior = zona_turno.select(F.col("beat").alias("vecino"), "turno",
                                    (F.col("semana_global") + 1).alias("semana_global"),
                                    F.col("incidentes").alias("incidentes_vecino"))
promedio_vecinos = (vecinos.select("beat", "vecino")
                    .join(vecino_anterior, "vecino")
                    .groupBy("beat", "turno", "semana_global")
                    .agg(F.avg("incidentes_vecino").alias("incidentes_vecinos_semana_anterior"),
                         F.count("*").alias("vecinos_con_dato")))

variables = variables.join(promedio_vecinos, ["beat", "turno", "semana_global"], "left")

### 2.4 Días festivos

Los 11 festivos federales de Estados Unidos, en su fecha real (no el día de descanso que se recorre cuando caen en fin de semana). Una semana puede tener 0, 1 o 2. Se conocen de antemano, así que usarlos no mete información del futuro.

In [ ]:
def n_esimo(anio, mes, dia_semana, n):
    """El n-ésimo día de la semana del mes (dia_semana: 0 = lunes); n = -1 es el último."""
    if n > 0:
        primero = date(anio, mes, 1)
        return primero + timedelta(days=(dia_semana - primero.weekday()) % 7 + 7 * (n - 1))
    ultimo = date(anio + (mes == 12), mes % 12 + 1, 1) - timedelta(days=1)
    return ultimo - timedelta(days=(ultimo.weekday() - dia_semana) % 7)


def festivos_federales(anio):
    lunes, jueves = 0, 3
    return {
        "Año Nuevo": date(anio, 1, 1),
        "Martin Luther King Jr.": n_esimo(anio, 1, lunes, 3),
        "Día de los Presidentes": n_esimo(anio, 2, lunes, 3),
        "Memorial Day": n_esimo(anio, 5, lunes, -1),
        "Juneteenth": date(anio, 6, 19),
        "Día de la Independencia": date(anio, 7, 4),
        "Labor Day": n_esimo(anio, 9, lunes, 1),
        "Día de Colón": n_esimo(anio, 10, lunes, 2),
        "Día de los Veteranos": date(anio, 11, 11),
        "Acción de Gracias": n_esimo(anio, 11, jueves, 4),
        "Navidad": date(anio, 12, 25),
    }


filas_festivos = [(dia.year, (dia.timetuple().tm_yday - 1) // 7, nombre)
                  for anio in range(PRIMER_ANIO, ANIO_PRUEBA + 1)
                  for nombre, dia in festivos_federales(anio).items()]
festivos = (spark.createDataFrame(filas_festivos, "anio int, semana int, festivo string")
            .filter("semana < 52")   # el 31 de diciembre queda fuera de las semanas de A3
            .groupBy("anio", "semana").agg(F.count("*").alias("festivos_semana")))

variables = (variables.join(festivos, ["anio", "semana"], "left")
             .fillna(0, ["festivos_semana"]))
print("Festivos de 2025:", ", ".join(f"{n} ({d:%d/%m})" for n, d in festivos_federales(2025).items()))

### 2.5 Clima (optimista)

Clima diario del aeropuerto O'Hare (reanálisis ERA5 de Open-Meteo), agregado a las mismas semanas de A3: temperatura media, lluvia total y nieve total. Se usa el clima **de la semana que se predice**. En la práctica no se conoce de antemano y habría que usar un pronóstico, así que este grupo mide cuánto ayudaría un pronóstico perfecto: es un techo, no un resultado que se pueda usar tal cual.

In [ ]:
clima_diario = spark.read.csv(RUTA_CLIMA, header=True, inferSchema=True)
clima = (clima_diario
         .withColumn("dia", F.to_date("fecha"))
         .withColumn("anio", F.year("dia"))
         .withColumn("semana", F.floor((F.dayofyear("dia") - 1) / 7).cast("int"))
         .filter("semana < 52")
         .groupBy("anio", "semana")
         .agg(F.round(F.avg("temperatura_media"), 2).alias("temperatura_media"),
              F.round(F.sum("lluvia_mm"), 2).alias("lluvia_mm"),
              F.round(F.sum("nieve_cm"), 2).alias("nieve_cm"),
              F.count("*").alias("dias_clima")))

assert clima.filter("dias_clima != 7").count() == 0, "Alguna semana no tiene sus 7 días de clima"
variables = variables.join(clima.drop("dias_clima"), ["anio", "semana"], "left")
print(f"{clima_diario.count():,} días de clima · {clima.count()} semanas")

### 2.6 Filas completas y verificación a mano

Una fila entra al modelo solo si tiene todas las variables. El retraso de 52 semanas y el promedio del último año piden un año completo de historia, así que el modelo empieza en la semana 4 de 2023 (semana global 56). Para que la comparación sea justa, **todas** las variantes, incluida la de la Fase 3, se entrenan con esas mismas filas.

La verificación repite la de la Fase 3: se toma el beat con más incidentes, el turno de 12 a 18 y la semana 30 de 2025, y cada variable nueva se recalcula directo de la tabla zona-turno, sin ventanas ni joins.

In [ ]:
GRUPOS = {
    "retrasos": ["incidentes_hace_2", "incidentes_hace_52", "promedio_52_semanas"],
    "estacionalidad": ["semana_seno", "semana_coseno"],
    "vecinos": ["incidentes_vecinos_semana_anterior"],
    "festivos": ["festivos_semana"],
    "clima (optimista)": ["temperatura_media", "lluvia_mm", "nieve_cm"],
}
NUEVAS = [v for grupo in GRUPOS.values() for v in grupo]

completas = (variables
             .filter((F.col("semanas_en_promedio_52") == 52) & (F.col("vecinos_con_dato") == VECINOS))
             .drop("semanas_en_promedio_52", "vecinos_con_dato")
             .withColumn("conjunto", F.when(F.col("semana_global") < CORTE_GLOBAL, "entrenamiento").otherwise("prueba"))
             .cache())

n_completas = completas.count()
primera = completas.agg(F.min("semana_global")).first()[0]
assert completas.filter(" OR ".join(f"{c} IS NULL" for c in VARIABLES_FASE3 + NUEVAS)).count() == 0
print(f"{n_completas:,} filas con todas las variables, desde la semana global {primera} "
      f"(semana {primera % 52} de {PRIMER_ANIO + primera // 52})")
(completas.groupBy("conjunto")
 .agg(F.countDistinct("semana_global").alias("semanas"),
      F.date_format(F.min("semana_inicio"), "dd/MM/yyyy").alias("desde"),
      F.date_format(F.date_add(F.max("semana_inicio"), 6), "dd/MM/yyyy").alias("hasta"),
      F.count("*").alias("filas"))
 .orderBy("conjunto").toPandas())

In [ ]:
beat_ej = zona_turno.groupBy("beat").agg(F.sum("incidentes").alias("t")).orderBy(F.desc("t")).first()["beat"]
turno_ej, semana_ej = 2, a_semana_global(ANIO_PRUEBA, 30)

serie = {r["semana_global"]: r["incidentes"] for r in
         zona_turno.filter((F.col("beat") == beat_ej) & (F.col("turno") == turno_ej)).collect()}
vecinos_ej = [r["vecino"] for r in vecinos.filter(F.col("beat") == beat_ej).collect()]
previa_vecinos = [r["incidentes"] for r in
                  zona_turno.filter(F.col("beat").isin(vecinos_ej) & (F.col("turno") == turno_ej)
                                    & (F.col("semana_global") == semana_ej - 1)).collect()]
fila = completas.filter((F.col("beat") == beat_ej) & (F.col("turno") == turno_ej)
                        & (F.col("semana_global") == semana_ej)).first()

a_mano = {
    "incidentes_hace_2": serie[semana_ej - 2],
    "incidentes_hace_52": serie[semana_ej - 52],
    "promedio_52_semanas": sum(serie[s] for s in range(semana_ej - 52, semana_ej)) / 52,
    "incidentes_vecinos_semana_anterior": sum(previa_vecinos) / len(previa_vecinos),
}
for nombre, valor in a_mano.items():
    assert abs(fila[nombre] - valor) < 1e-9, nombre
print(f"Beat {beat_ej}, turno {fila['turno_etiqueta']}, semana 30 de {ANIO_PRUEBA} (desde el {fila['semana_inicio']:%d/%m/%Y})")
print(f"  Vecinos: {', '.join(map(str, vecinos_ej))}")
for nombre, valor in a_mano.items():
    print(f"  {nombre} = {fila[nombre]:.2f} · contado a mano: {valor:.2f} ✔")
print(f"  Festivos en la semana: {fila['festivos_semana']} · temperatura media: {fila['temperatura_media']} °C")

¿Cuánto se relaciona cada variable con los incidentes de la semana que se predice? (en las filas de entrenamiento)

In [ ]:
entrenamiento_corr = completas.filter(F.col("conjunto") == "entrenamiento")
pd.DataFrame([(grupo, v, round(entrenamiento_corr.stat.corr(v, "incidentes"), 3))
              for grupo, lista in [("fase 3", VARIABLES_FASE3), *GRUPOS.items()] for v in lista],
             columns=["grupo", "variable", "correlación con incidentes"])

### 2.7 Guardar la tabla con las variables nuevas

En curated, junto a la tabla de la Fase 3: `zona_turno_variables_2022_2025`.

In [ ]:
df_salida = completas.select("beat", "district", "turno", "turno_etiqueta", "anio", "semana", "semana_global",
                             "semana_inicio", *VARIABLES_FASE3, *NUEVAS, "incidentes", "conjunto").coalesce(4)
if LOCAL:
    df_salida.write.mode("overwrite").parquet(RUTA_SALIDA)
    tabla = spark.read.parquet(RUTA_SALIDA)
else:
    (df_salida.write.mode("overwrite").format("parquet").option("path", RUTA_SALIDA).saveAsTable(TABLA_SALIDA))
    tabla = spark.table(TABLA_SALIDA)

assert tabla.count() == n_completas
zona_turno.unpersist(); completas.unpersist()
print(f"{n_completas:,} filas guardadas en {RUTA_SALIDA}" + ("" if LOCAL else f" · tabla {TABLA_SALIDA}"))

## 3. Referencias y métricas

Las mismas métricas de la Fase 3 en las semanas 42 a 51 de 2025: RMSE, MAE y R², y captura y PAI sobre el 10 % de zonas-turno que se señalarían cada semana. Además de las dos referencias de la Fase 3 hay una tercera regla simple: el **promedio de las últimas 52 semanas**. Si un modelo no le gana a esa regla, la variable nueva no está aportando nada que un promedio no dé.

In [ ]:
df_entrenamiento = tabla.filter(F.col("conjunto") == "entrenamiento").cache()
df_prueba = tabla.filter(F.col("conjunto") == "prueba").cache()
print(f"Entrenamiento: {df_entrenamiento.count():,} filas · prueba: {df_prueba.count():,} filas")

ZONAS_SENALADAS = 0.10


def metricas(df, prediccion):
    """RMSE, MAE y R² de una columna de predicción contra los incidentes reales."""
    evaluador = RegressionEvaluator(labelCol="incidentes", predictionCol=prediccion)
    return {nombre: evaluador.evaluate(df, {evaluador.metricName: metrica})
            for nombre, metrica in (("RMSE", "rmse"), ("MAE", "mae"), ("R²", "r2"))}


def tasa_captura(df, prediccion):
    """Porcentaje de los incidentes que caen, cada semana, en el 10 % de zonas-turno con la predicción más alta."""
    orden = Window.partitionBy("semana_global").orderBy(F.desc(prediccion), "beat", "turno")
    senaladas = (df.withColumn("puesto", F.row_number().over(orden))
                 .withColumn("zonas", F.count("*").over(Window.partitionBy("semana_global")))
                 .filter(F.col("puesto") <= F.ceil(F.col("zonas") * ZONAS_SENALADAS)))
    return senaladas.agg(F.sum("incidentes")).first()[0] / df.agg(F.sum("incidentes")).first()[0]


def mae_semanal(df, prediccion):
    """MAE de cada semana de prueba."""
    return (df.groupBy("semana").agg(F.mean(F.abs(F.col("incidentes") - F.col(prediccion))).alias("mae"))
            .toPandas().set_index("semana")["mae"].sort_index())


def evaluar(df, prediccion, nombre):
    captura = tasa_captura(df, prediccion)
    return {"modelo": nombre, **metricas(df, prediccion), "captura (%)": captura * 100, "PAI": captura / ZONAS_SENALADAS}


REFERENCIAS = {"Promedio histórico": "promedio_historico", "Promedio de 52 semanas": "promedio_52_semanas",
               "Promedio de 4 semanas": "promedio_4_semanas"}
referencias = [evaluar(df_prueba, columna, nombre) for nombre, columna in REFERENCIAS.items()]
mae_historico = mae_semanal(df_prueba, "promedio_historico")
captura_maxima = tasa_captura(df_prueba, "incidentes")
print(f"Captura máxima posible: {captura_maxima:.1%}")
pd.DataFrame(referencias).set_index("modelo").round(3)

## 4. ¿Cuánto aporta cada grupo?

Random Forest con los mismos hiperparámetros que eligió la Fase 3 con cuatro años (`maxDepth = 8`, `numTrees = 50`), para que la única diferencia entre variantes sean las variables. Se prueba la base de la Fase 3, la base más cada grupo por separado, todas las variables honestas juntas y, al final, todas más el clima.

A diferencia de la Fase 3 no se escalan las variables: los árboles cortan por umbrales y el escalado no les cambia nada. El turno y el distrito entran con one-hot, como antes.

In [ ]:
CATEGORICAS = ["turno_etiqueta", "district"]
codificacion = Pipeline(stages=[
    StringIndexer(inputCols=CATEGORICAS, outputCols=[f"{c}_idx" for c in CATEGORICAS]),
    OneHotEncoder(inputCols=[f"{c}_idx" for c in CATEGORICAS], outputCols=[f"{c}_ohe" for c in CATEGORICAS]),
]).fit(df_entrenamiento)
entrenamiento_cod = codificacion.transform(df_entrenamiento).cache()
prueba_cod = codificacion.transform(df_prueba).cache()
entrenamiento_cod.count(); prueba_cod.count()
ONE_HOT = [f"{c}_ohe" for c in CATEGORICAS]


def entrenar(numericas, entrenamiento=None, evaluacion=None, maxDepth=8, numTrees=50):
    """Entrena Random Forest con esas variables numéricas más turno y distrito. Devuelve el modelo, las predicciones y los segundos."""
    entrenamiento = entrenamiento_cod if entrenamiento is None else entrenamiento
    evaluacion = prueba_cod if evaluacion is None else evaluacion
    ensamblador = VectorAssembler(inputCols=numericas + ONE_HOT, outputCol="features")
    inicio = time.time()
    modelo = RandomForestRegressor(featuresCol="features", labelCol="incidentes", maxDepth=maxDepth,
                                   numTrees=numTrees, seed=42).fit(ensamblador.transform(entrenamiento))
    return modelo, modelo.transform(ensamblador.transform(evaluacion)), time.time() - inicio


HONESTAS = [v for g, lista in GRUPOS.items() if g != "clima (optimista)" for v in lista]
VARIANTES = {"Fase 3 (base)": VARIABLES_FASE3,
             **{f"base + {grupo}": VARIABLES_FASE3 + lista for grupo, lista in GRUPOS.items()},
             "base + todas sin clima": VARIABLES_FASE3 + HONESTAS,
             "base + todas con clima (optimista)": VARIABLES_FASE3 + NUEVAS}

filas, predicciones = [], {}
for nombre, numericas in VARIANTES.items():
    spark.sparkContext.setJobDescription(f"#21 · {nombre}")
    _, prediccion, segundos = entrenar(numericas)
    prediccion = prediccion.select("beat", "turno", "semana", "semana_global", "incidentes", "prediction").cache()
    predicciones[nombre] = prediccion
    gana = int((mae_semanal(prediccion, "prediction") < mae_historico).sum())
    filas.append({**evaluar(prediccion, "prediction", nombre), "semanas con menor MAE que el promedio histórico": gana,
                  "entrenamiento (s)": round(segundos, 1)})
    print(f"{nombre}: MAE {filas[-1]['MAE']:.3f} · gana en {gana} de 10 semanas · {segundos:.1f} s")
spark.sparkContext.setJobDescription(None)

comparacion_grupos = pd.DataFrame(filas).set_index("modelo")
comparacion_grupos.round(3)

In [ ]:
mae_hist = referencias[0]["MAE"]
fig, ax = plt.subplots(figsize=(9, 4.5))
orden = comparacion_grupos["MAE"].sort_values(ascending=False)
colores = ["#b8b8b8" if "clima" in n else "#2a9d8f" if n != "Fase 3 (base)" else "#2a78d6" for n in orden.index]
ax.barh(orden.index, orden.values, color=colores)
ax.axvline(mae_hist, color="#333333", linestyle="--", label=f"Promedio histórico ({mae_hist:.3f})")
ax.axvline(referencias[1]["MAE"], color="#e07b28", linestyle=":", label=f"Promedio de 52 semanas ({referencias[1]['MAE']:.3f})")
ax.set_xlim(min(orden.min(), mae_hist) - 0.02, max(orden.max(), mae_hist) + 0.01)
ax.set(title="#21 · MAE en las semanas de prueba por grupo de variables (Random Forest)", xlabel="MAE (menos es mejor)")
ax.legend(loc="lower right", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 5. Ajuste de hiperparámetros con ventanas de tiempo

En la Fase 3 los hiperparámetros se eligieron con una sola partición (semanas 34 a 41). Aquí se usan tres ventanas de validación de 8 semanas cada una, todas dentro de 2025 y antes de la prueba: 18 a 25, 26 a 33 y 34 a 41. Para cada ventana el modelo entrena con todo lo anterior a ella, igual que se usaría en la realidad. Gana la combinación con menor MAE promedio en las tres.

Se ajusta la variante honesta con todas las variables (sin clima).

In [ ]:
VENTANAS = [(18, 25), (26, 33), (34, 41)]
REJILLA = [(profundidad, arboles) for profundidad in (8, 12) for arboles in (50, 100)]

resultados_ventanas = []
for desde, hasta in VENTANAS:
    inicio_v, fin_v = a_semana_global(ANIO_PRUEBA, desde), a_semana_global(ANIO_PRUEBA, hasta)
    ajuste = entrenamiento_cod.filter(F.col("semana_global") < inicio_v)
    validacion = entrenamiento_cod.filter(F.col("semana_global").between(inicio_v, fin_v))
    for profundidad, arboles in REJILLA:
        _, prediccion, _ = entrenar(VARIABLES_FASE3 + HONESTAS, ajuste, validacion, profundidad, arboles)
        resultados_ventanas.append({"ventana": f"{desde}-{hasta}", "maxDepth": profundidad, "numTrees": arboles,
                                    "MAE": metricas(prediccion, "prediction")["MAE"]})

tabla_ventanas = (pd.DataFrame(resultados_ventanas)
                  .pivot_table(index=["maxDepth", "numTrees"], columns="ventana", values="MAE"))
tabla_ventanas["promedio"] = tabla_ventanas.mean(axis=1)
tabla_ventanas = tabla_ventanas.sort_values("promedio")
MEJOR_PROFUNDIDAD, MEJORES_ARBOLES = tabla_ventanas.index[0]
print(f"Mejor combinación: maxDepth = {MEJOR_PROFUNDIDAD}, numTrees = {MEJORES_ARBOLES}")
tabla_ventanas.round(4)

## 6. Modelo final contra las referencias

In [ ]:
spark.sparkContext.setJobDescription("#21 · modelo final")
modelo_final, pred_final, segundos_final = entrenar(VARIABLES_FASE3 + HONESTAS, maxDepth=int(MEJOR_PROFUNDIDAD),
                                                     numTrees=int(MEJORES_ARBOLES))
spark.sparkContext.setJobDescription(None)
pred_final = pred_final.select("beat", "turno", "semana", "semana_global", "incidentes", "prediction").cache()

mae_final = mae_semanal(pred_final, "prediction")
gana_final = int((mae_final < mae_historico).sum())
fila_final = {**evaluar(pred_final, "prediction", "Random Forest con variables nuevas (ajustado)"),
              "semanas con menor MAE que el promedio histórico": gana_final, "entrenamiento (s)": round(segundos_final, 1)}

final = pd.DataFrame([fila_final, {**comparacion_grupos.loc["Fase 3 (base)"].to_dict(), "modelo": "Random Forest de la Fase 3 (mismas filas)"},
                      *referencias]).set_index("modelo").sort_values("MAE")
print(f"Random Forest ajustado ({MEJOR_PROFUNDIDAD} de profundidad, {MEJORES_ARBOLES} árboles) · "
      f"menor MAE que el promedio histórico en {gana_final} de 10 semanas · "
      f"{fila_final['captura (%)'] / 100 / captura_maxima:.0%} de la captura máxima posible")
final.round(3).fillna("—")

In [ ]:
semana_a_semana = pd.DataFrame({"Random Forest con variables nuevas": mae_final, "Promedio histórico": mae_historico,
                                "Promedio de 52 semanas": mae_semanal(df_prueba, "promedio_52_semanas")})
fig, ax = plt.subplots(figsize=(9, 4))
for nombre, color in zip(semana_a_semana.columns, ["#2a9d8f", "#333333", "#e07b28"]):
    ax.plot(semana_a_semana.index, semana_a_semana[nombre], marker="o", color=color, label=nombre)
ax.set(title="#21 · MAE semana a semana en las semanas de prueba de 2025", xlabel="Semana del año", ylabel="MAE")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()
semana_a_semana.round(3)

### Importancia de las variables

Cuánto pesa cada variable en las decisiones de los árboles del modelo final (suman 1). Turno y distrito se suman en un solo valor cada uno.

In [ ]:
nombres = VARIABLES_FASE3 + HONESTAS
etiquetas = codificacion.stages[0].labelsArray
nombres += [f"{c}={v}" for c, valores in zip(CATEGORICAS, etiquetas) for v in valores[:-1]]
importancia = pd.Series(modelo_final.featureImportances.toArray(), index=nombres)
grupo_de = {v: g for g, lista in GRUPOS.items() for v in lista} | {v: "fase 3" for v in VARIABLES_FASE3}
por_variable = importancia.groupby(importancia.index.map(
    lambda n: "turno" if n.startswith("turno_etiqueta=") else "distrito" if n.startswith("district=") else n)).sum()
(por_variable.sort_values(ascending=False).round(3).to_frame("importancia")
 .assign(grupo=lambda t: t.index.map(lambda n: grupo_de.get(n, n))))

## 7. Guardar las métricas

La tabla de la sección 4 y el modelo final, en `curated/metricas_variables_2022_2025`, para el reporte.

In [ ]:
METRICAS = {"RMSE": "rmse", "MAE": "mae", "R²": "r2", "captura (%)": "captura", "PAI": "pai",
            "semanas con menor MAE que el promedio histórico": "semanas_mejor_que_historico"}
tabla_metricas = (pd.concat([comparacion_grupos, pd.DataFrame([fila_final]).set_index("modelo"),
                             pd.DataFrame(referencias).set_index("modelo")])[list(METRICAS)]
                  .rename(columns=METRICAS).astype(float).reset_index())
spark.createDataFrame(tabla_metricas).coalesce(1).write.mode("overwrite").parquet(RUTA_METRICAS)
print("Métricas guardadas en", RUTA_METRICAS)

## Conclusiones

*Se completan con los resultados de la corrida en el clúster.*

---

## Evidencia de ejecución en el clúster

In [ ]:
sc = spark.sparkContext
print(f"Aplicación: {sc.appName} · {sc.applicationId}")
print(f"Master: {sc.master} · ejecutores activos: {sc._jsc.sc().getExecutorMemoryStatus().size() - 1}")
if not LOCAL:
    print(f"Historial: http://localhost:18080/history/{sc.applicationId}/jobs/")

conf_hadoop = sc._jsc.hadoopConfiguration()
for capa, ruta in [("raw", RUTA_CLIMA), ("curated", RUTA_SALIDA), ("curated", RUTA_METRICAS)]:
    ruta_hadoop = spark._jvm.org.apache.hadoop.fs.Path(ruta)
    resumen = ruta_hadoop.getFileSystem(conf_hadoop).getContentSummary(ruta_hadoop)
    print(f"{capa:<10} {resumen.getFileCount():>3} archivos · {resumen.getLength() / 2**20:6.1f} MiB · {ruta}")

## Cierre

In [ ]:
spark.stop()   # libera la sesión: en la instancia solo cabe una sesión de Spark a la vez